In [ ]:
import sys, platform, os
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import pdb
import healpy as hp
from astropy.io import fits
import time
import math
from scipy import interpolate
import treecorr
import pickle as pk
import configparser
from pixell import enmap
from pixell import reproject
import ast
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy import units as u
import kmeans_radec
import h5py as h5
import argparse
import gc
import dill

%load_ext autoreload
%autoreload 2



## Run the DES shear processing and save the ra, dec and shear values of the DES source galaxies in a separate file, would be faster to measure the correlations:


In [ ]:
# download the file to open and process the DES files: https://github.com/des-science/DESY3Cats/tree/main
# Also download all the public DES Y3 catalogs at https://desdr-server.ncsa.illinois.edu/despublic/y3a2_files/y3kp_cats/
# Make sure you re-link the sompz to v0.5 version:
# with h5py.File('DESY3_indexcat.h5', 'r+') as f: 
    # del f['catalog/sompz']
    # f['catalog/sompz'] = h5py.ExternalLink('DESY3_sompz_v0.50.h5', 'catalog/sompz')

sdir = '/global/cfs/cdirs/des/data_actxdes/des_data/'
save_fname = sdir + 'cat_DES_shearcat_all_dump_nzfix_Feb25.pk'

# check if the file already exists
if os.path.exists(save_fname):
    cat_DES = dill.load(open(save_fname,'rb'))
    print('cat_ACT already exists')
else:
    import DESY3Cats as destest
    import sys, os

    destest_dict_ = {
        'output_exists' : True,
        'use_mpi'       : False,
        'source'        : 'hdf5',
        'dg'            : 0.01
        }



    # # Populates a full destest yaml dict for each catalog selection based on the limited catalog input info provided in the common cats.yaml file
    def create_destest_yaml( params, name, cal_type, group, table, select_path ):
        """
        Creates the input dictionary structure from a passed dictionary rather than reading froma yaml file.
        """
        destest_dict = destest_dict_.copy()
        destest_dict['load_cache'] = params['load_cache']
        destest_dict['output'] = params['output']
        destest_dict['name'] = name
        destest_dict['filename'] = '/global/cfs/cdirs/lsst/www/shivamp/data_actxdes/Y3_cats/DESY3_indexcat.h5'
        destest_dict['param_file'] = params['param_file']
        destest_dict['cal_type'] = cal_type
        destest_dict['group'] = group
        destest_dict['table'] = table
        destest_dict['select_path'] = select_path
        destest_dict['e'] = ['e_1','e_2']
        destest_dict['Rg'] = ['R11','R22']
        destest_dict['w'] = 'weight'
        return destest_dict
    # Build selector (and calibrator) classes from destest for the catalog.
    def load_catalog(pipe_params, name, cal_type, group, table, select_path, inherit=None, return_calibrator=None):
        """
        Loads data access and calibration classes from destest for a given yaml setup file.
        """
        # Input yaml file defining catalog
        params = create_destest_yaml(pipe_params, name, cal_type, group, table, select_path)
        # Load destest source class to manage access to file
        source = destest.H5Source(params)
        # Load destest selector class to manage access to data in a structured way
        if inherit is None:
            sel = destest.Selector(params,source)
        else:
            sel = destest.Selector(params,source,inherit=inherit)
        # Load destest calibrator class to manage calibration of the catalog
        if return_calibrator is not None:
            cal = return_calibrator(params,sel)
            return sel, cal
        else:
            return sel
        
    # Read yaml file that defines all the catalog selections used
    params = yaml.load(open('cats.yaml'), Loader=yaml.SafeLoader)
    params['param_file'] = 'cats.yaml'
    # Source catalog
    source_selector, source_calibrator = load_catalog(
        params, 'mcal', 'mcal', params['source_group'], params['source_table'], params['source_path'], return_calibrator=destest.MetaCalib)

    gold_selector = load_catalog(
        params, 'gold', 'mcal', params['gold_group'], params['gold_table'], params['gold_path'], inherit=source_selector)

    pz_selector = load_catalog(
        params, 'pz', 'mcal', params['pz_group'], params['pz_table'], params['pz_path'], inherit=source_selector)


    cat_DES = dict()
    for i in range(4):
        print (i)
        pzbin = pz_selector.get_col('bhat') # 5-tuple for metacal (un)sheared versions                                    
        mask = [pzbin[j] == i for j in range(5)] # First tomographic bin    
        print(mask)
        g1 = source_selector.get_col('e_1')[0][mask[0]]
        g2 = source_selector.get_col('e_2')[0][mask[0]]
        ra = gold_selector.get_col('ra')[0][mask[0]]
        dec = gold_selector.get_col('dec')[0][mask[0]]
        print(g1, g2, ra, dec)
        wa = source_calibrator.calibrate('e_1', mask=mask,weight_only=True) 
        R1,c,w = source_calibrator.calibrate('e_1',mask=mask) 
        R2,c,w = source_calibrator.calibrate('e_2',mask=mask)
        g1 =(g1 - np.mean(g1*wa)/np.mean(wa))/R1
        g2 =(g2 - np.mean(g2*wa)/np.mean(wa))/R2

        theta_datapoint_all, phi_datapoint_all = eq2ang(ra, dec)
        ind_datapoints = hp.ang2pix(nside, theta_datapoint_all, phi_datapoint_all)
        int_ind = np.in1d(ind_datapoints, ind_unmasked)
        ind = np.where(int_ind == True)[0]
        ind = np.arange(len(ra))
        
        print('orignal numbers : ' + str(len(ra)))    
        cat_DES[i] = [g1[ind],g2[ind], ra[ind],dec[ind], 1, w[ind]]

    sdir = '/global/cfs/cdirs/des/data_actxdes/des_data/'
    dill.dump(cat_DES,open(sdir + 'cat_DES_shearcat_all_dump_nzfix_Feb25.pk','wb'))


In [ ]:
def ang2eq(theta, phi):
    ra = phi * 180. / np.pi
    dec = 90. - theta * 180. / np.pi
    return ra, dec

def eq2ang(ra, dec):
    phi = ra * np.pi / 180.
    theta = (np.pi / 2.) - dec * (np.pi / 180.)
    return theta, phi

# Constants and configuration
NSIDE = 4096
MASK_PATH = '/global/cfs/cdirs/des/data_actxdes/mask_updated_ACTDR6xDESY3.fits'
YMAP_DIR = '/global/cfs/cdirs/act/data/synced_maps/ILC_MAPS/20230606/'
SAVE_DIR = '/global/cfs/cdirs/lsst/www/shivamp/cosmosis-standard-library/ACTxDESY3/src/run_measurements/results_nzfix_Feb25/'
NJK = 250
JK_OBJ_FILENAME = f'jkobj_DES_njk_{NJK}_v27Mar23.pk'
NUM_CORES = 256
RAD2DEG = 180. / np.pi

# TreeCorr configuration
treecorr_config = {
    'nbins': 20,
    'min_sep': 2.5,
    'max_sep': 250.,
    'sep_units': 'arcmin',
    'bin_slop': 0.0,
    'num_threads': NUM_CORES,
    'var_method': 'jackknife',
    'verbose': 0
}

# List of deprojections
deproj_list = ['cib_1p7_dBeta', 'None', 'cib_1p4', 'cib_1p7', 'cib_2p0', 'cib_1p4_dBeta', 'cib_2p0_dBeta']

# File mapping for different deprojections
deproj_to_file = {
    'None': 'ilc_SZ_yy.fits',
    'cib_1p0': 'ilc_SZ_deproj_cib_1.0_10.7_yy.fits',
    'cib_1p2': 'ilc_SZ_deproj_cib_1.2_10.7_yy.fits',
    'cib_1p4': 'ilc_SZ_deproj_cib_1.4_10.7_yy.fits',
    'cib_1p6': 'ilc_SZ_deproj_cib_1.6_10.7_yy.fits',
    'cib_1p8': 'ilc_SZ_deproj_cib_1.8_10.7_yy.fits',
    'cib_2p0': 'ilc_SZ_deproj_cib_2.0_10.7_yy.fits',
    'cib_1p0_dBeta': 'ilc_SZ_deproj_cib_cibdBeta_1.0_10.7_yy.fits',
    'cib_1p2_dBeta': 'ilc_SZ_deproj_cib_cibdBeta_1.2_10.7_yy.fits',
    'cib_1p4_dBeta': 'ilc_SZ_deproj_cib_cibdBeta_1.4_10.7_yy.fits',
    'cib_1p6_dBeta': 'ilc_SZ_deproj_cib_cibdBeta_1.6_10.7_yy.fits',
    'cib_1p7_dBeta': 'ilc_SZ_deproj_cib_cibdBeta_1.7_10.7_yy.fits',
    'cib_1p8_dBeta': 'ilc_SZ_deproj_cib_cibdBeta_1.8_10.7_yy.fits',
    'cib_2p0_dBeta': 'ilc_SZ_deproj_cib_cibdBeta_2.0_10.7_yy.fits'
}

# Read mask and get unmasked pixels
print("Reading mask...")
mask_f = hp.read_map(MASK_PATH)
hp.mollview(mask_f)
ind_unmasked = np.where(mask_f > 0.9)[0]
del mask_f
gc.collect()

# Process each deprojection
for deproj in deproj_list:
    print(f'Processing {deproj}')
    
    # Get y-map file path
    y_file = os.path.join(YMAP_DIR, deproj_to_file.get(deproj, ''))
    if not os.path.exists(y_file):
        print(f"File not found: {y_file}")
        continue
    
    # Read y-map
    print('Opening y-map and mask')
    imapy = enmap.read_map(y_file)
    decs, ras = imapy.posmap()
    ra_y = np.array(RAD2DEG * ras.flatten())
    dec_y = np.array(RAD2DEG * decs.flatten())
    ymap_truth = np.array(imapy.flatten())
    
    # Filter by mask
    theta_datapoint_all, phi_datapoint_all = eq2ang(ra_y, dec_y)
    ind_datapoints = hp.ang2pix(NSIDE, theta_datapoint_all, phi_datapoint_all)
    selection_f = np.where(np.in1d(ind_datapoints, ind_unmasked))[0]
    
    del theta_datapoint_all, phi_datapoint_all, ind_datapoints
    gc.collect()
    
    # Process each redshift bin
    for jb in range(4):
        file_suffix = f'zbin_{jb}_njk_{NJK}'
        save_fname = os.path.join(
            SAVE_DIR, 
            f'kg_act_deprojects_{deproj}_vnew_wbeam_pixell_{file_suffix}_healpix{NSIDE}_theta_2p5_250_vFeb25.pk'
        )
        
        if os.path.isfile(save_fname):
            print(f'Skipping bin {jb+1}, file already exists')
            continue
            
        print(f'Processing bin: {jb+1}')
        jk_obj_path = os.path.join(SAVE_DIR, JK_OBJ_FILENAME)
        
        # Create catalogs for correlation
        cat_b = treecorr.Catalog(
            ra=ra_y[selection_f], 
            dec=dec_y[selection_f], 
            ra_units='deg', 
            dec_units='deg',
            k=ymap_truth[selection_f], 
            patch_centers=jk_obj_path if os.path.isfile(jk_obj_path) else None,
            npatch=NJK if not os.path.isfile(jk_obj_path) else None
        )
        
        # Save jackknife centers if needed
        if not os.path.isfile(jk_obj_path):
            cat_b.write_patch_centers(jk_obj_path)
            
        cat_a = treecorr.Catalog(
            ra=cat_DES[jb][2], 
            dec=cat_DES[jb][3], 
            g1=cat_DES[jb][0], 
            g2=cat_DES[jb][1],
            w=cat_DES[jb][5],
            ra_units='deg', 
            dec_units='deg', 
            patch_centers=jk_obj_path
        ) 
        
        gc.collect()
        
        # Calculate KG correlation
        kg = treecorr.KGCorrelation(**treecorr_config)
        print('Computing KG correlation')
        kg.process(cat_b, cat_a, num_threads=NUM_CORES)
        print('Computing KG covariance')
        cov_kg = kg.estimate_cov('jackknife')
        
        # Extract results
        xi_kg = kg.xi
        r_kg = np.exp(kg.meanlogr)
        
        print(f"r_kg: {r_kg}")
        print(f"xi_kg: {xi_kg}")
        print(f"Diagonal errors: {np.sqrt(np.diag(cov_kg))}")
        
        # Save results
        save_data = {
            # 'kg': kg,
            'xi_kg': xi_kg,
            'r_kg': r_kg, 
            'cov_dy': cov_kg, 
            'njk': NJK
        }
        
        dill.dump(save_data, open(save_fname, 'wb'))
        print(f"Saved results to {save_fname}")




In [ ]:
nbins = 4
for deproj in deproj_list:
    data_deproj = {}
    for jbin in range(nbins):
        df = dill.load(open(SAVE_DIR + '/kg_act_deprojects_' + deproj + '_vnew_wbeam_pixell_zbin_' + str(jbin) + '_njk_250_healpix4096_theta_2p5_250_vFeb25.pk','rb'))   
        rkg = df['r_kg']
        covkg = df['cov_dy']
        xikg = df['xi_kg']
        data_deproj[jbin] = {'rkg':rkg, 'xikg':xikg, 'covkg':covkg}
    dill.dump(data_deproj, open(SAVE_DIR + '/kg_act_deprojects_' + deproj + '_vnew_wbeam_pixell_zbin_all_njk_250_healpix4096_theta_2p5_250_vFeb25.pk','wb'))

    

